# 🔍 KOSIS 통계 RAG 5대 핵심 지표 진단 및 Human-in-the-Loop 평가 워크벤치 (v2)

이 노트북은 단일 쿼리에 대해 파이프라인의 치명적 병목 구간(특히 **Module 3 뷰 슬라이싱/수치 추출** 및 **Module 4~5 컨텍스트 조립·변화율 계산·환각 검증**)을 추적하고, 아래 **5대 핵심 지표**를 **`PASS` / `PARTIAL` / `FAIL`** 3단계로 평가·기록합니다.

### 📊 5대 핵심 평가 지표 (`PASS` | `PARTIAL` | `FAIL`)
1. **검색 적합성 (`eval_retrieval`)**: 질문의 도메인/시점/분류에 맞는 View를 정확히 가져왔는가?
2. **원본 수치 추출 (`eval_extraction`)**: Parquet/CSV에서 시작·최근·1위·최하위 수치를 소계(`동부` 등) 혼입 없이 정확히 뽑았는가?
3. **Context 충분성 (`eval_context`)**: LLM에게 전달되는 프롬프트에 원시 수치와 사전 계산된 `facts`(변화량/변화율)가 누락 없이 포함되었는가?
4. **변화율 응답 완성도 (`eval_completeness`)**: 질문이 요구한 추이·변화량·변화율(%)·순위 비교를 계산 오류 없이 완성도 있게 답했는가?
5. **자동 환각 검증 (`eval_hallucination`)**: `verify_numbers`가 암산 오류(예: `782,259` → `882,259`)나 환각 수치를 정확히 잡아냈는가?

In [1]:
# [Cell 1] 환경 설정, API 키 재검증 및 모듈 임포트
import json
import os
import re
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, HTML, Markdown
import pandas as pd
import duckdb

# .env 강제 리로드 및 키 공백/따옴표 정제 (HTTP 401 방지)
load_dotenv(Path.cwd() / ".env", override=True)
if os.getenv("HCX_API_KEY"):
    os.environ["HCX_API_KEY"] = os.getenv("HCX_API_KEY").strip().strip("'\"")

import kosis_views as kv
import kosis_rag_qa as qa
from golden_prompts import JUDGE_SCHEMA, JUDGE_SYSTEM

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', 120)

DATA_DIR = Path("kosis_data")
VIEWS_DIR = Path("kosis_data/rag_views_golden") if Path("kosis_data/rag_views_golden").exists() else Path("kosis_data/rag_views")
VARIANT = "golden_v1" if (DATA_DIR / "index" / "golden_v1" / "meta.json").exists() else "base"

qa.VIEWS_DIR = VIEWS_DIR
idx = qa.Index(VARIANT)
cat_df = pd.read_csv(DATA_DIR / "catalog.csv", dtype=str) if (DATA_DIR / "catalog.csv").exists() else pd.DataFrame()

key_preview = (os.getenv("HCX_API_KEY") or "NONE")[:8] + "..."
print(f"✅ 데이터 폴더 : {DATA_DIR.resolve()}")
print(f"✅ 뷰 폴더     : {VIEWS_DIR.resolve()}")
print(f"✅ 로드된 인덱스: {VARIANT} (총 문서 수: {len(idx.ids):,}개) | HCX_API_KEY: {key_preview}")

✅ 데이터 폴더 : C:\Project\Llion\project\t4core-stat-agent\kosis_data
✅ 뷰 폴더     : C:\Project\Llion\project\t4core-stat-agent\kosis_data\rag_views_golden
✅ 로드된 인덱스: golden_v1 (총 문서 수: 314개) | HCX_API_KEY: nv-415ea...


--- 
### [Step 0] 테스트할 단일 쿼리(Query) 입력
- 시계열 변화율 케이스나 시도별 순위 비교 케이스 등 분석할 질문을 `USER_QUERY`에 입력하세요.

In [2]:
# [Cell 2] 분석할 단일 쿼리 지정
USER_QUERY = "시도내이동-시군구내 추이 (1970~2025)에서 1971년 대비 2025년 수치는 얼마이고 변화량과 변화율은 어떻게 변했어?"
# USER_QUERY = "2010 시도별 단독주택 기준 가장 수치가 높은 1위와 가장 낮은 최하위는 각각 어디야?"

TOP_K = 5
USE_REWRITE = True

print(f"📝 입력 질의: {USER_QUERY}")

📝 입력 질의: 시도내이동-시군구내 추이 (1970~2025)에서 1971년 대비 2025년 수치는 얼마이고 변화량과 변화율은 어떻게 변했어?


--- 
### [Module 1] 질의 재작성(Query Rewrite), 형태소 토큰화 및 조건 추출 진단
- API 키 오류(`HTTP 401`) 발생 시 중단되지 않고 자동으로 원문 질의(`USE_REWRITE=False`)로 폴백(Fallback)합니다.

In [3]:
# [Cell 3] Module 1 실행 (API 401 에러 시 자동 폴백 포함)
rewrite_status = "SUCCESS"
if USE_REWRITE:
    try:
        rewritten_q, sub_questions = qa.rewrite(USER_QUERY)
    except Exception as e:
        rewrite_status = f"FALLBACK_NO_REWRITE ({e})"
        rewritten_q, sub_questions = USER_QUERY, []
else:
    rewritten_q, sub_questions = USER_QUERY, []

tokens_orig = qa.tokenize(USER_QUERY)
tokens_rw = qa.tokenize(rewritten_q)

extracted_regions = qa.mentioned_regions(USER_QUERY)
extracted_years = list(dict.fromkeys(re.findall(r"(19\d{2}|20\d{2})\s*년?", USER_QUERY)))
extracted_sex = [w for w in ("남자", "여자") if w in USER_QUERY or w[0] + "성" in USER_QUERY]
asks_rate_or_change = bool(re.search(r"변화|증감|추이|비교|변했|차이|율", USER_QUERY))

display(Markdown(f"""
### 📊 Module 1 진단 결과
- **원본 질문**: `{USER_QUERY}`
- **재작성 상태**: `{rewrite_status}`
- **재작성 검색어 (`rewritten`)**: `{rewritten_q}` | **하위 질문**: `{sub_questions}`
- **BM25 형태소 토큰**: `{tokens_rw}`
- **규칙 기반 추출 조건**: 지역=`{extracted_regions}`, 연도=`{extracted_years}`, 성별=`{extracted_sex}`, 변화/추이요구=`{asks_rate_or_change}`
"""))


### 📊 Module 1 진단 결과
- **원본 질문**: `시도내이동-시군구내 추이 (1970~2025)에서 1971년 대비 2025년 수치는 얼마이고 변화량과 변화율은 어떻게 변했어?`
- **재작성 상태**: `SUCCESS`
- **재작성 검색어 (`rewritten`)**: `시도내 이동 및 시군구 내 이동의 추이에 대한 1971년부터 2025년까지의 수치, 변화량 및 변화율 조회` | **하위 질문**: `['1971년의 시도내 이동 및 시군구 내 이동 수치', '2025년의 시도내 이동 및 시군구 내 이동 수치', '1971년부터 2025년까지의 시도내 이동 및 시군구 내 이동의 변화량']`
- **BM25 형태소 토큰**: `['시도', '내', '이동', '시군구', '내', '이동', '추이', '대하', '1971', '년', '2025', '년', '수치', '변화', '량', '변화', '조회']`
- **규칙 기반 추출 조건**: 지역=`[]`, 연도=`['1970', '2025', '1971']`, 성별=`[]`, 변화/추이요구=`True`


--- 
### [Module 2] 하이브리드 검색(BM25 + Dense + RRF) 및 참고 자료 링크 확인 (`검색 적합성` 점검)

In [4]:
# [Cell 4] Module 2 하이브리드 검색 (임베딩 API 실패 시 BM25 단독 폴백 지원)
queries = [rewritten_q] + sub_questions
agg = {}
weights = {q: 1.0 for q in queries}
if USER_QUERY not in weights:
    weights[USER_QUERY] = 0.5

for q, w in weights.items():
    try:
        search_res = idx.search(q, k=TOP_K * 2)
    except Exception as e:
        print(f"⚠ Dense 임베딩 호출 실패 ({e}) -> BM25 단독 검색으로 대체합니다.")
        bm_only = idx.search_bm25(q, k=TOP_K * 2)
        search_res = [{"id": b["id"], "score": b["score"], "title": b["title"],
                       "doc_type": idx.by_id[b["id"]].get("doc_type"),
                       "bm25_rank": r_i, "dense_rank": None, "dense_sim": float("nan")} for r_i, b in enumerate(bm_only)]
    for r, h in enumerate(search_res):
        agg.setdefault(h["id"], {"h": h, "s": 0.0})["s"] += w / (qa.RRF_K + r + 1)

hits = [v["h"] | {"fused": round(v["s"], 5)} for v in sorted(agg.values(), key=lambda x: -x["s"])][:TOP_K]

def get_reference_links(rec):
    src = rec.get("source", {})
    org_id = src.get("org_id", "")
    tbl_id = src.get("tbl_id", "")
    tbl_key = src.get("table_key", f"{org_id}_{tbl_id}")
    kosis_url = src.get("kosis_url") or f"https://kosis.kr/statHtml/statHtml.do?orgId={org_id}&tblId={tbl_id}"
    category = rec.get("category") or rec.get("survey_nm", "")
    raw_csv_path = DATA_DIR / "raw" / kv.safe(category) / f"{tbl_key}.csv"
    if not raw_csv_path.exists() and not cat_df.empty:
        m = cat_df[(cat_df["org_id"] == str(org_id)) & (cat_df["tbl_id"] == str(tbl_id))]
        if not m.empty:
            raw_csv_path = DATA_DIR / "raw" / kv.safe(m.iloc[0]["category"]) / f"{tbl_key}.csv"
    parquet_path = VIEWS_DIR / src.get("file", f"processed/{tbl_key}.parquet")
    return {"kosis_url": kosis_url, "raw_csv": raw_csv_path, "parquet": parquet_path}

search_rows = []
for rank, h in enumerate(hits, 1):
    rec = idx.by_id[h["id"]]
    links = get_reference_links(rec)
    f = rec.get("filters", {})
    search_rows.append({
        "순위": rank,
        "문서ID": h["id"],
        "뷰 제목": h["title"],
        "조사명": rec.get("category", ""),
        "수록기간": f"{f.get('prd_from', '')}~{f.get('prd_to', '')}",
        "Fused점수": h["fused"],
        "BM25순위": h["bm25_rank"],
        "Dense순위": f"#{h['dense_rank']} ({h['dense_sim']})",
        "KOSIS_웹링크": f'<a href="{links["kosis_url"]}" target="_blank">🔗 KOSIS 통계표 열기</a>',
        "원본_CSV": f"✅ {links['raw_csv'].as_posix()}" if links["raw_csv"].exists() else f"❌ 없음"
    })

display(HTML(pd.DataFrame(search_rows).to_html(escape=False, index=False)))

순위,문서ID,뷰 제목,조사명,수록기간,Fused점수,BM25순위,Dense순위,KOSIS_웹링크,원본_CSV
1,view::101_DT_1B26001_A03::A::03::trend,시도내이동-시군구내 추이 (1970~2025),국내인구이동통계,1971~2025,0.07377,0,#0 (0.8467),🔗 KOSIS 통계표 열기,✅ kosis_data/raw/국내인구이동통계/101_DT_1B26001_A03.csv
2,view::101_DT_1B26001_A03::Q::18::trend,시도내이동-시군구내 추이 (1970 1/4분기~2026 2/4분기),국내인구이동통계,197101~202602,0.07258,1,#2 (0.7991),🔗 KOSIS 통계표 열기,✅ kosis_data/raw/국내인구이동통계/101_DT_1B26001_A03.csv
3,view::101_DT_1B26001_A03::A::04::trend,시도내이동-시군구간 전입 추이 (1970~2025),국내인구이동통계,1970~2025,0.07143,2,#1 (0.7994),🔗 KOSIS 통계표 열기,✅ kosis_data/raw/국내인구이동통계/101_DT_1B26001_A03.csv
4,view::101_DT_1B26001_A03::Q::19::trend,시도내이동-시군구간 전입 추이 (1970 1/4분기~2026 2/4분기),국내인구이동통계,197001~202602,0.07031,3,#3 (0.7602),🔗 KOSIS 통계표 열기,✅ kosis_data/raw/국내인구이동통계/101_DT_1B26001_A03.csv
5,view::101_DT_1B26001_A03::A::05::items_trend,시군구/연령(5세)별 이동자수: 항목별 추이,국내인구이동통계,1970~2025,0.06832,6,#5 (0.7349),🔗 KOSIS 통계표 열기,✅ kosis_data/raw/국내인구이동통계/101_DT_1B26001_A03.csv


--- 
### [Module 3] 뷰 슬라이싱 및 원본 수치 추출 정밀 대조 (`원본 수치 추출` 점검)
- **치명적 점검 포인트**:
  1. View의 `facts`에 **시작점, 최근점, 변화량(`-782,259명`), 변화율(`-27.2%`)** 또는 **1위/최하위**가 정확히 계산되어 있는가?
  2. 분류 코드 사전(`Profile`)에 `동부/읍부/면부` 같은 중간 소계가 일반 행정구역으로 섞여 있거나, `서울특별시(depth=0)`와 `전국(depth=0)`의 계층 깊이가 꼬여 있지 않은가?

In [5]:
# [Cell 5] Module 3: View 내부 Facts(사전 계산된 변화량/변화율) 및 원본 CSV/Parquet 대조
INSPECT_RANK = 1
target_hit = hits[INSPECT_RANK - 1]
target_rec = idx.by_id[target_hit["id"]]
if target_rec.get("doc_type") == "table":
    vids = [v for v in target_rec.get("view_ids", []) if v in idx.by_id]
    if vids:
        target_rec = idx.by_id[vids[0]]

links = get_reference_links(target_rec)
prof = qa._profile(target_rec["source"]["table_key"])

display(Markdown(f"""
### 📌 점검 대상 View (Rank #{INSPECT_RANK})
- **View ID**: `{target_rec['id']}` | **제목**: `{target_rec.get('title')}`
- **설명**: {target_rec.get('description')}
- **🌐 KOSIS 원본 링크**: [{links['kosis_url']}]({links['kosis_url']})
- **📁 로컬 원본 CSV**: `{links['raw_csv'].as_posix()}` (존재: `{links['raw_csv'].exists()}`)
"""))

facts_list = target_rec.get("facts", [])
facts_dict = {f["label"]: f for f in facts_list}
print("[1] Module 3가 원본 Parquet에서 추출해 둔 핵심 수치 (facts):")
display(pd.DataFrame(facts_list))

print("[2] View 미리보기 (preview 상위/하위 5행):")
pv = target_rec.get("preview", {})
if pv.get("rows"):
    pv_df = pd.DataFrame(pv["rows"], columns=pv.get("columns"))
    display(pd.concat([pv_df.head(5), pv_df.tail(5)]))


### 📌 점검 대상 View (Rank #1)
- **View ID**: `view::101_DT_1B26001_A03::A::03::trend` | **제목**: `시도내이동-시군구내 추이 (1970~2025)`
- **설명**: 국내인구이동통계의 「시군구/연령(5세)별 이동자수」 통계표에서 시도내이동-시군구내(명)의 시점별 추이를 1971~2025 기준으로 정리한 자료입니다. 고정 조건: 행정구역(시군구)별: 전국; 연령별: 계.
- **🌐 KOSIS 원본 링크**: [http://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1B26001_A03&lang_mode=ko&vw_cd=MT_ZTITLE&list_id=A34_3&conn_path=I4](http://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1B26001_A03&lang_mode=ko&vw_cd=MT_ZTITLE&list_id=A34_3&conn_path=I4)
- **📁 로컬 원본 CSV**: `kosis_data/raw/국내인구이동통계/101_DT_1B26001_A03.csv` (존재: `True`)


[1] Module 3가 원본 Parquet에서 추출해 둔 핵심 수치 (facts):


,label,value,unit,text
0,시작,2.878091e+06,명,"1971 2,878,091명"
1,최근,2.095832e+06,명,"2025 2,095,832명"
2,변화량,-7.822590e+05,명,"1971 대비 2025 -782,259명 감소"
3,변화율,-2.717980e+01,%,변화율 -27.2%
4,최대,6.616530e+06,명,"최대 1988 6,616,530명"
5,최소,2.095832e+06,명,"최소 2025 2,095,832명"
6,직전대비,-6.500200e+00,%,직전 시점(2024) 대비 -6.5%


[2] View 미리보기 (preview 상위/하위 5행):


,period_label,value
0,1971,2878091.0
1,1972,2619367.0
2,1973,3435573.0
3,1974,3709343.0
4,1975,6143486.0
50,2021,2649108.0
51,2022,2173548.0
52,2023,2154739.0
53,2024,2241537.0
54,2025,2095832.0


--- 
### [Module 4] 컨텍스트 조립(`context_block`) 및 `facts` 유실 진단 (`Context 충분성` 점검)
- **🚨 왜 `In[43]`~`In[44]`에서 변화율 응답이 실패하고 암산 오차(`882,259`)가 났는가?**
  - `kosis_rag_qa.py`의 `context_block(rec, max_rows, cond)` 코드를 보면, 질문에서 연도(`1971`, `2025`)가 추출되어 `cond`가 활성화되면 **235행에서 `return "\n".join(lines)`로 조기 종료(Early Return)** 해버립니다.
  - 그 결과, 바로 위 `[Cell 5]`의 `facts`에 이미 정확하게 계산되어 있던 **`변화량: -782,259명 감소`, `변화율: -27.2%`가 컨텍스트에서 통째로 날아가고**, `1971 | 2,878,091`과 `2025 | 2,095,832` 두 줄만 LLM에게 전달되었습니다!
  - 아래 셀에서 **기존 컨텍스트(As-Is)**와 **`facts` 유실 여부를 자동으로 진단**합니다.

In [6]:
# [Cell 6] Module 4: 조건 슬라이싱 및 프롬프트 컨텍스트(context_block) 충분성 진단
ctx_recs = []
for h in hits:
    rec = idx.by_id[h["id"]]
    if rec.get("doc_type") == "table":
        vids = [v for v in rec.get("view_ids", []) if v in idx.by_id][:1]
        if vids and vids[0] not in [c["id"] for c in ctx_recs]:
            rec = idx.by_id[vids[0]]
    if rec["id"] not in [c["id"] for c in ctx_recs] and rec.get("doc_type") == "view":
        ctx_recs.append(rec)

conds, srcs = {}, {}
for r in ctx_recs:
    try:
        src, applied, missing = qa.apply_conditions(r, USER_QUERY)
        crow = qa.condition_rows(src, max_rows=40)[1] if applied else None
    except Exception as e:
        src, applied, missing, crow = r["source"], {}, [f"조건 적용 실패: {e.__class__.__name__}"], None
    conds[r["id"]], srcs[r["id"]] = (applied, missing, crow), src

ctx_recs.sort(key=lambda r: (len(conds[r["id"]][1]) > 0, -len(conds[r["id"]][0])))
ctx_recs = ctx_recs[:4]

full_context_str = "\n\n".join(qa.context_block(r, 40, conds[r["id"]]) for r in ctx_recs)

# [진단] View에는 facts(변화량/변화율)가 있는데, 조립된 full_context_str에서는 누락되었는지 확인
top_ctx_rec = ctx_recs[0]
has_facts_in_view = bool(top_ctx_rec.get("facts"))
has_facts_in_prompt = "핵심 수치(기본 단면):" in full_context_str or "변화율" in full_context_str

if has_facts_in_view and not has_facts_in_prompt:
    display(Markdown("""⚠️ **[Module 4 구조적 병목 감지!]**
- View 내부(`facts`)에는 **`변화량`과 `변화율(%)`이 이미 계산되어 존재**하지만, `qa.context_block()`이 `질문 조건 적용(cond)` 분기에서 **Early Return**하면서 `facts` 줄을 프롬프트에 넣지 않고 생략했습니다!
- 이로 인해 원시 수치 2개는 전달되었으나(`Context 충분성: PASS/PARTIAL`), LLM이 직접 뺄셈을 하다가 암산 실수(`882,259`)를 범하고 변화율 응답을 누락(`변화율 응답 완성도: FAIL`)하게 됩니다."""))

print("=== [LLM에게 실제로 전달되는 프롬프트 컨텍스트 전문] ===")
print("-" * 80)
print(full_context_str)
print("-" * 80)

⚠️ **[Module 4 구조적 병목 감지!]**
- View 내부(`facts`)에는 **`변화량`과 `변화율(%)`이 이미 계산되어 존재**하지만, `qa.context_block()`이 `질문 조건 적용(cond)` 분기에서 **Early Return**하면서 `facts` 줄을 프롬프트에 넣지 않고 생략했습니다!
- 이로 인해 원시 수치 2개는 전달되었으나(`Context 충분성: PASS/PARTIAL`), LLM이 직접 뺄셈을 하다가 암산 실수(`882,259`)를 범하고 변화율 응답을 누락(`변화율 응답 완성도: FAIL`)하게 됩니다.

=== [LLM에게 실제로 전달되는 프롬프트 컨텍스트 전문] ===
--------------------------------------------------------------------------------
[자료 id=view::101_DT_1B26001_A03::A::03::trend]
제목: 시도내이동-시군구내 추이 (1970~2025)
출처: 자료: 국가데이터처,「국내인구이동통계」
설명: 국내인구이동통계의 「시군구/연령(5세)별 이동자수」 통계표에서 시도내이동-시군구내(명)의 시점별 추이를 1971~2025 기준으로 정리한 자료입니다. 고정 조건: 행정구역(시군구)별: 전국; 연령별: 계.
질문 조건 적용: 시점=['1970', '2025', '1971', '2025']
조건 적용 데이터: period_label | c1_name | c2_name | itm_nm | value | unit
1971 | 전국 | 계 | 시도내이동-시군구내 | 2,878,091 | 명
2025 | 전국 | 계 | 시도내이동-시군구내 | 2,095,832 | 명
주의: 1. 1986년 이전에는 전국 전출자료에 국외 및 기타가 포함된 경우가 있어 시도별 합이 전국계(0)와 일치하지 않을 수 있음 2. 시군구의 구는 특광역시의 자치구로서 수원, 성남, 안양, 부천, 안산, 고양, 용인, 화성, 청주,  천안, 전주, 포항, 창원시의 일반 / 행정구역 개편(예: 세종시 신설 2012, 군위군 대구 편입 2023)으로 시점별 지역 범위가 다를 수 있음 / 일부 값은 통계부호(-, X, … 등)로 결측 처리됨 (0이 아님)

[자료 id=view::101_DT_1B26001_A03::Q::18::trend]
제목: 시도내이동-시군구내 추이 (1970 1/4분기~2026 2/4분기)
출처: 자료: 국가데이터처,「국내인구이동통계」
설명: 국내인구이동통계의 「시군구/연령(5세)별 이동자수」 통계표에서 시도내이동-시군구내(명)의 시점별 추이를 1971 1/4분기~2026 2/4분기 기

--- 
### [Module 5] LLM 답변 생성 및 수치·변화율·환각 정밀 대조 (`변화율 응답 완성도` & `자동 환각 검증`)

In [7]:
# [Cell 7] Module 5: LLM 답변 생성 (또는 이전 실행 캐시 입력) 및 5대 지표 자동 분석
try:
    ans_res = qa.answer(USER_QUERY, variant=VARIANT, k=TOP_K, use_rewrite=USE_REWRITE, save=False, idx=idx)
except Exception as e:
    print(f"⚠ LLM API 호출 실패 ({e}). 최근 실행 로그(In[44]) 예시로 시뮬레이션합니다.")
    ans_res = {
        "question": USER_QUERY,
        "rewritten": rewritten_q,
        "answer": "1971년 대비 2025년의 시도내이동-시군구내 인구 이동 수는 약 882,259명 감소했습니다.",
        "summary": "1971년 대비 2025년의 시도내이동-시군구내 인구 이동 수가 감소했습니다.",
        "key_points": [
            "1971년 시도내이동-시군구내 인구 이동 수: 2,878,091명",
            "2025년 시도내이동-시군구내 인구 이동 수: 2,095,832명",
            "변화량: 약 882,259명 감소"
        ],
        "used_ids": [top_ctx_rec["id"]],
        "chart_id": top_ctx_rec["id"],
        "insufficient": False,
        "unverified_numbers": qa.verify_numbers("1971 2025 2,878,091 2,095,832 882,259", full_context_str),
        "caveats": top_ctx_rec.get("caveats", []),
        "condition_matched": True
    }

combined_ans_text = " ".join([ans_res["answer"], ans_res.get("summary", "")] + list(ans_res.get("key_points") or []))
ans_num_set = qa.nums(combined_ans_text)

# --- 5대 핵심 지표 자동 판정 로직 (PASS / PARTIAL / FAIL) ---
# 1) 검색 적합성
auto_retrieval = "PASS" if top_ctx_rec.get("doc_type") == "view" and ans_res.get("condition_matched", True) else "PARTIAL"

# 2) 원본 수치 추출 (시작/최근 또는 1위/최하위 원본 수치가 AI 응답/핵심수치에 정확히 들어갔는지)
base_labels = [l for l in ("시작", "최근", "1위", "최하위") if l in facts_dict]
extracted_ok = [l for l in base_labels if qa.nums(str(facts_dict[l]["value"])) & ans_num_set]
if len(base_labels) > 0 and len(extracted_ok) == len(base_labels):
    auto_extraction = "PASS"
elif len(extracted_ok) > 0:
    auto_extraction = "PARTIAL"
else:
    auto_extraction = "FAIL"

# 3) Context 충분성
auto_context = "PASS" if (len(full_context_str) > 100 and not ans_res["insufficient"]) else "FAIL"

# 4) 변화율 응답 완성도 (변화량 및 변화율 % 수치가 정확히 답변되었는지)
has_diff_in_facts = "변화량" in facts_dict
has_rate_in_facts = "변화율" in facts_dict
diff_correct = bool(qa.nums(str( abs(facts_dict["변화량"]["value"]) )) & ans_num_set) if has_diff_in_facts else True
rate_correct = bool(qa.nums(str( abs(facts_dict["변화율"]["value"]) )) & ans_num_set) if has_rate_in_facts else True

if has_diff_in_facts or has_rate_in_facts:
    if diff_correct and rate_correct:
        auto_completeness = "PASS"
    elif diff_correct or rate_correct:
        auto_completeness = "PARTIAL"
    else:
        auto_completeness = "FAIL"
else:
    auto_completeness = auto_extraction

# 5) 자동 환각 검증 (unverified_numbers 탐지 상태 분석)
unv = ans_res.get("unverified_numbers") or []
if not unv:
    auto_hallucination = "PASS"
elif auto_extraction == "PASS" and not diff_correct:
    # 원본 수치는 맞았으나 LLM이 직접 뺄셈/나눗셈을 하다가 틀린 숫자가 unverified_numbers에 잡힌 경우 -> PARTIAL
    auto_hallucination = "PARTIAL"
else:
    auto_hallucination = "FAIL"

auto_metrics = {
    "검색 적합성": auto_retrieval,
    "원본 수치 추출": auto_extraction,
    "Context 충분성": auto_context,
    "변화율 응답 완성도": auto_completeness,
    "자동 환각 검증": auto_hallucination,
}

display(Markdown(f"""
### 💬 Module 5 답변 및 5대 지표 자동 분석 결과
- **AI 답변 (`answer`)**: {ans_res['answer']}
- **AI 핵심 수치 (`key_points`)**: `{ans_res['key_points']}`
- **실제 Facts 정답**: `{ {k: v['text'] for k, v in facts_dict.items()} }`
- **미검증 숫자 탐지 (`unverified_numbers`)**: `{unv}` (실제 변화량 `{facts_dict.get('변화량', {}).get('text', '-')}` vs AI 계산값 `{unv}`)

| 평가 지표 | 자동 판정 (Auto-Rule) | 진단 근거 |
| :--- | :---: | :--- |
| **검색 적합성** | **{auto_metrics['검색 적합성']}** | Top-1 View (`{top_ctx_rec['id']}`) 매칭 완료 |
| **원본 수치 추출** | **{auto_metrics['원본 수치 추출']}** | 기준 시점 원본 수치 매칭: `{extracted_ok}` / `{base_labels}` |
| **Context 충분성** | **{auto_metrics['Context 충분성']}** | 조건 적용 데이터 전달 완료 (단, `facts` 변화량/변화율 줄은 Early Return으로 생략됨) |
| **변화율 응답 완성도** | **{auto_metrics['변화율 응답 완성도']}** | 변화량 일치=`{diff_correct}`, 변화율(%) 포함=`{rate_correct}` |
| **자동 환각 검증** | **{auto_metrics['자동 환각 검증']}** | `verify_numbers`가 LLM 암산 오차(`{unv}`)를 미검증 숫자로 검출함 |
"""))


### 💬 Module 5 답변 및 5대 지표 자동 분석 결과
- **AI 답변 (`answer`)**: 1971년과 2025년의 시도내이동-시군구내 이동자수는 각각 2,878,091명과 2,095,832명으로 나타났습니다. 이로 인해 변화량은 약 -782,259명이 되었으며, 변화율은 약 -27.14% 감소했습니다.
- **AI 핵심 수치 (`key_points`)**: `['2,878,091', '2,095,832', '-782,259', '-27.14%']`
- **실제 Facts 정답**: `{'시작': '1971 2,878,091명', '최근': '2025 2,095,832명', '변화량': '1971 대비 2025 -782,259명 감소', '변화율': '변화율 -27.2%', '최대': '최대 1988 6,616,530명', '최소': '최소 2025 2,095,832명', '직전대비': '직전 시점(2024) 대비 -6.5%'}`
- **미검증 숫자 탐지 (`unverified_numbers`)**: `['27.14', '782,259']` (실제 변화량 `1971 대비 2025 -782,259명 감소` vs AI 계산값 `['27.14', '782,259']`)

| 평가 지표 | 자동 판정 (Auto-Rule) | 진단 근거 |
| :--- | :---: | :--- |
| **검색 적합성** | **PASS** | Top-1 View (`view::101_DT_1B26001_A03::A::03::trend`) 매칭 완료 |
| **원본 수치 추출** | **PASS** | 기준 시점 원본 수치 매칭: `['시작', '최근']` / `['시작', '최근']` |
| **Context 충분성** | **PASS** | 조건 적용 데이터 전달 완료 (단, `facts` 변화량/변화율 줄은 Early Return으로 생략됨) |
| **변화율 응답 완성도** | **PASS** | 변화량 일치=`True`, 변화율(%) 포함=`True` |
| **자동 환각 검증** | **FAIL** | `verify_numbers`가 LLM 암산 오차(`['27.14', '782,259']`)를 미검증 숫자로 검출함 |


--- 
### [Module 6] 최종 HUMAN 평가 (`PASS` / `PARTIAL` / `FAIL` 3단계 정의 및 저장)
- 아래 셀에서 **5대 세부 지표**와 **최종 HUMAN 종합 평가(`FINAL_HUMAN_VERDICT`)**를それぞれ **`"PASS"`, `"PARTIAL"`, `"FAIL"`** 중 하나로 지정하고 실행하면, 화면에 비교표가 출력되고 `single_query_debug_log.csv`에 모든 지표가 개별 컬럼으로 저장됩니다.

In [8]:
# [Cell 8] 5대 지표별 Human 평가 및 최종 판정(PASS / PARTIAL / FAIL) 저장

VALID_GRADES = {"PASS", "PARTIAL", "FAIL"}

# 👇 1. 5대 세부 지표에 대한 평가를 PASS / PARTIAL / FAIL 중 하나로 설정하세요 (기본값은 위 자동 판정값)
HUMAN_METRICS = {
    "검색 적합성": "PASS",           # PASS | PARTIAL | FAIL
    "원본 수치 추출": "PASS",        # PASS | PARTIAL | FAIL
    "Context 충분성": "PASS",       # PASS | PARTIAL | FAIL
    "변화율 응답 완성도": "FAIL",    # PASS | PARTIAL | FAIL
    "자동 환각 검증": "PARTIAL",     # PASS | PARTIAL | FAIL
}

# 👇 2. 최종 HUMAN 종합 평가를 PASS / PARTIAL / FAIL 중 하나로 설정하세요
FINAL_HUMAN_VERDICT = "PARTIAL"      # "PASS" | "PARTIAL" | "FAIL"

TARGET_MODULE_TO_FIX = (
    "Module 4 & 5 (kosis_rag_qa.py: context_block 조건 적용 시에도 facts의 변화량/변화율을 함께 제공하여 LLM 암산 오류 방지)"
)
HUMAN_COMMENT = (
    "검색과 1971년(2,878,091명)·2025년(2,095,832명) 원본 수치 추출 및 Context 전달은 정상(PASS)이나, "
    "context_block에서 조건 적용 시 facts(변화량 -782,259명, 변화율 -27.2%)가 생략되어 "
    "LLM이 직접 뺄셈을 수행하다 882,259명으로 오산함(변화율 응답 완성도 FAIL, 자동 환각 검증 PARTIAL)."
)

# 입력값 검증
for k_m, v_m in {**HUMAN_METRICS, "최종_HUMAN_평가": FINAL_HUMAN_VERDICT}.items():
    assert v_m in VALID_GRADES, f"'{k_m}'의 값('{v_m}')은 반드시 {VALID_GRADES} 중 하나여야 합니다."

# 참고 링크 정리
ref_links_md, ref_links_plain = [], []
for uid in ans_res.get("used_ids", [top_ctx_rec["id"]]):
    if uid in idx.by_id:
        r_used = idx.by_id[uid]
        l_info = get_reference_links(r_used)
        ref_links_md.append(
            f"- **{r_used['title']}** (`{uid}`)\n"
            f"  - 🌐 KOSIS 웹 통계표: [{l_info['kosis_url']}]({l_info['kosis_url']})\n"
            f"  - 📁 로컬 Raw CSV: `{l_info['raw_csv'].as_posix()}` (존재: {l_info['raw_csv'].exists()})"
        )
        ref_links_plain.append(f"{uid} | KOSIS: {l_info['kosis_url']} | CSV: {l_info['raw_csv'].as_posix()}")

top_facts_str = ", ".join(f["text"] for f in (top_ctx_rec.get("facts") or []))

# 리포트 출력
report_md = f"""
---
### 📋 단일 쿼리 5대 지표 진단 및 HUMAN 최종 평가 리포트
- **입력** : {USER_QUERY}
- **출력** : {ans_res['answer']}
  - *(AI 추출 핵심수치)* : `{', '.join(ans_res.get('key_points', []))}`
- **원본 뷰 요약 기준 (Facts)** : `{top_facts_str}`
- **미검증/오산 수치 (`unverified_numbers`)** : `{ans_res.get('unverified_numbers', [])}`
- **참고 자료 링크** :
{chr(10).join(ref_links_md)}

#### 📊 세부 지표 및 최종 평가 (`PASS` / `PARTIAL` / `FAIL`)
| 구분 | 검색 적합성 | 원본 수치 추출 | Context 충분성 | 변화율 응답 완성도 | 자동 환각 검증 | **최종 HUMAN 평가** |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **자동 진단** | `{auto_metrics['검색 적합성']}` | `{auto_metrics['원본 수치 추출']}` | `{auto_metrics['Context 충분성']}` | `{auto_metrics['변화율 응답 완성도']}` | `{auto_metrics['자동 환각 검증']}` | - |
| **HUMAN 판정** | **{HUMAN_METRICS['검색 적합성']}** | **{HUMAN_METRICS['원본 수치 추출']}** | **{HUMAN_METRICS['Context 충분성']}** | **{HUMAN_METRICS['변화율 응답 완성도']}** | **{HUMAN_METRICS['자동 환각 검증']}** | 🔥 **{FINAL_HUMAN_VERDICT}** |

- **판정 근거 및 코멘트** : {HUMAN_COMMENT}
- **수정 대상 모듈** : `{TARGET_MODULE_TO_FIX}`
---
"""
display(Markdown(report_md))

# CSV에 5대 세부 지표와 최종 HUMAN 평가(PASS/PARTIAL/FAIL)를 각각 컬럼으로 저장
log_entry = {
    "입력": USER_QUERY,
    "재작성_검색어": ans_res.get("rewritten", ""),
    "출력": ans_res.get("answer", ""),
    "뷰_요약기준(Facts)": top_facts_str,
    "미검증_수치(unverified)": ", ".join(ans_res.get("unverified_numbers", [])),
    "검색_적합성": HUMAN_METRICS["검색 적합성"],
    "원본_수치_추출": HUMAN_METRICS["원본 수치 추출"],
    "Context_충분성": HUMAN_METRICS["Context 충분성"],
    "변화율_응답_완성도": HUMAN_METRICS["변화율 응답 완성도"],
    "자동_환각_검증": HUMAN_METRICS["자동 환각 검증"],
    "최종_HUMAN_평가": FINAL_HUMAN_VERDICT,
    "사람_코멘트": HUMAN_COMMENT,
    "수정_대상_모듈": TARGET_MODULE_TO_FIX,
    "참고자료_링크_및_CSV": " || ".join(ref_links_plain),
}

log_csv_path = VIEWS_DIR / "single_query_debug_log_v2.csv"
df_log = pd.DataFrame([log_entry])
df_log.to_csv(log_csv_path, mode="a", header=not log_csv_path.exists(), index=False, encoding="utf-8-sig")
print(f"💾 5대 세부 지표 및 최종 HUMAN 평가({FINAL_HUMAN_VERDICT})가 저장되었습니다 → {log_csv_path}")
display(pd.read_csv(log_csv_path).tail(5))


---
### 📋 단일 쿼리 5대 지표 진단 및 HUMAN 최종 평가 리포트
- **입력** : 시도내이동-시군구내 추이 (1970~2025)에서 1971년 대비 2025년 수치는 얼마이고 변화량과 변화율은 어떻게 변했어?
- **출력** : 1971년과 2025년의 시도내이동-시군구내 이동자수는 각각 2,878,091명과 2,095,832명으로 나타났습니다. 이로 인해 변화량은 약 -782,259명이 되었으며, 변화율은 약 -27.14% 감소했습니다.
  - *(AI 추출 핵심수치)* : `2,878,091, 2,095,832, -782,259, -27.14%`
- **원본 뷰 요약 기준 (Facts)** : `1971 2,878,091명, 2025 2,095,832명, 1971 대비 2025 -782,259명 감소, 변화율 -27.2%, 최대 1988 6,616,530명, 최소 2025 2,095,832명, 직전 시점(2024) 대비 -6.5%`
- **미검증/오산 수치 (`unverified_numbers`)** : `['27.14', '782,259']`
- **참고 자료 링크** :
- **시도내이동-시군구내 추이 (1970~2025)** (`view::101_DT_1B26001_A03::A::03::trend`)
  - 🌐 KOSIS 웹 통계표: [http://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1B26001_A03&lang_mode=ko&vw_cd=MT_ZTITLE&list_id=A34_3&conn_path=I4](http://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1B26001_A03&lang_mode=ko&vw_cd=MT_ZTITLE&list_id=A34_3&conn_path=I4)
  - 📁 로컬 Raw CSV: `kosis_data/raw/국내인구이동통계/101_DT_1B26001_A03.csv` (존재: True)

#### 📊 세부 지표 및 최종 평가 (`PASS` / `PARTIAL` / `FAIL`)
| 구분 | 검색 적합성 | 원본 수치 추출 | Context 충분성 | 변화율 응답 완성도 | 자동 환각 검증 | **최종 HUMAN 평가** |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **자동 진단** | `PASS` | `PASS` | `PASS` | `PASS` | `FAIL` | - |
| **HUMAN 판정** | **PASS** | **PASS** | **PASS** | **FAIL** | **PARTIAL** | 🔥 **PARTIAL** |

- **판정 근거 및 코멘트** : 검색과 1971년(2,878,091명)·2025년(2,095,832명) 원본 수치 추출 및 Context 전달은 정상(PASS)이나, context_block에서 조건 적용 시 facts(변화량 -782,259명, 변화율 -27.2%)가 생략되어 LLM이 직접 뺄셈을 수행하다 882,259명으로 오산함(변화율 응답 완성도 FAIL, 자동 환각 검증 PARTIAL).
- **수정 대상 모듈** : `Module 4 & 5 (kosis_rag_qa.py: context_block 조건 적용 시에도 facts의 변화량/변화율을 함께 제공하여 LLM 암산 오류 방지)`
---


💾 5대 세부 지표 및 최종 HUMAN 평가(PARTIAL)가 저장되었습니다 → kosis_data\rag_views_golden\single_query_debug_log_v2.csv


,입력,재작성_검색어,출력,뷰_요약기준(Facts),미검증_수치(unverified),검색_적합성,원본_수치_추출,Context_충분성,변화율_응답_완성도,자동_환각_검증,최종_HUMAN_평가,사람_코멘트,수정_대상_모듈,참고자료_링크_및_CSV
0,시도내이동-시군구내 추이 (1970~2025)에서 1971년 대비 2025년 수치는 얼마이고 변화량과 변화율은 어떻게 변했어?,"시도내 이동 및 시군구 내 이동의 추이에 대한 1971년부터 2025년까지의 수치, 변화량 및 변화율 확인","1971년과 2025년의 시도내이동-시군구내 이동자수는 각각 2,878,091명과 2,095,832명으로 나타났습니다. 이로 인해 변화량은 약 -782,259명이 되었으며, 변화율은 약 -27.14% 감소했습니다.","1971 2,878,091명, 2025 2,095,832명, 1971 대비 2025 -782,259명 감소, 변화율 -27.2%, 최대 1988 6,616,530명, 최소 2025 2,095,832명, 직전 ...","27.14, 782,259",PASS,PASS,PASS,FAIL,PARTIAL,PARTIAL,"검색과 1971년(2,878,091명)·2025년(2,095,832명) 원본 수치 추출 및 Context 전달은 정상(PASS)이나, context_block에서 조건 적용 시 facts(변화량 -782,25...",Module 4 & 5 (kosis_rag_qa.py: context_block 조건 적용 시에도 facts의 변화량/변화율을 함께 제공하여 LLM 암산 오류 방지),view::101_DT_1B26001_A03::A::03::trend | KOSIS: http://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1B26001_A03&...
